# VoltesFive | HarborCart Phase 2: Model Development, Training & Iterative Tuning
Group: VoltesFive | Section: BSCS-3B | Instructor: Simon Gerard Granil

**Data note:** this notebook continues from the Phase 1 outputs built on the UCI Online Retail dataset (a UK retailer, GBP), used as a proxy for the fictional HarborCart. Nothing here is actual HarborCart or Philippine customer data.

**How to run (Google Colab):** run the Phase 1 notebook first, then run every cell here in order. If `harborcart_outputs/` is not in the session, you will be asked to upload `harborcart_phase1_outputs.zip`.

**What this notebook does (CRISP-DM: Modeling):**
1. **Descriptive:** correlation matrix, **k-Means** customer segmentation (RFM), **association rules** (mlxtend) and a rule **network** (networkx).
2. **Predictive:** will a customer purchase again in the final 90 days? Logistic Regression, Decision Tree and a small Neural Network (MLP), compared against a baseline.
3. **Experiment log:** every run (hyperparameters, split, metrics) is recorded in `Model_Experiment_Log.xlsx`.
4. **Artifacts:** `model_artifacts.pkl`.

All numbers are produced by running the code; none are typed in.

In [ ]:
import subprocess, sys
try:
    import mlxtend
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mlxtend'])

import json, pickle, zipfile, warnings, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import sklearn
from IPython.display import display
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from mlxtend.frequent_patterns import apriori, association_rules
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
RANDOM_STATE = 42
P1 = Path('harborcart_outputs'); P2 = Path('phase2_outputs'); P2.mkdir(exist_ok=True)
LOG = []   # experiment log

def log_run(exp_id, task, model, params, split, **metrics):
    LOG.append({'exp_id': exp_id, 'task': task, 'model': model,
                'params': json.dumps(params, default=str), 'split': split, **metrics})
print('Ready. scikit-learn', sklearn.__version__)

## 1. Load Phase 1 outputs

In [ ]:
if not (P1 / 'customer_rfm.csv').exists():
    from google.colab import files
    up = files.upload()
    z = [n for n in up if n.lower().endswith('.zip')]
    assert z, 'Upload harborcart_phase1_outputs.zip created by the Phase 1 notebook.'
    P1.mkdir(exist_ok=True)
    with zipfile.ZipFile(z[0]) as zf:
        zf.extractall(P1)

rfm = pd.read_csv(P1 / 'customer_rfm.csv', dtype={'CustomerID': 'string'}, parse_dates=['FirstPurchase', 'LastPurchase'])
tx = pd.read_csv(P1 / 'rfm_eligible_transactions.csv',
                 dtype={'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
meta = pd.read_csv(P1 / 'analysis_metadata.csv')
reference_date = pd.Timestamp(meta.loc[0, 'reference_date'])
print(f'Customers: {len(rfm):,} | eligible transaction lines: {len(tx):,} | reference date: {reference_date.date()}')
assert rfm['CustomerID'].is_unique

## 2. Feature preparation and correlation matrix
RFM variables are typically right-skewed with a few very large customers (e.g., wholesale buyers). **Decision:** keep extreme customers (they are real customers and distort revenue analysis if dropped), apply `log1p` and standardize before distance-based clustering. Spearman correlation is used because it does not assume linearity.

In [ ]:
FEATS = ['Recency', 'Frequency', 'Monetary']
EXTRA = ['AverageOrderValue', 'CustomerTenure', 'UniqueProducts']
skew = pd.DataFrame({'raw_skew': rfm[FEATS + EXTRA].skew(), 'log1p_skew': np.log1p(rfm[FEATS + EXTRA]).skew()})
display(skew.round(2))

q1, q3 = rfm['Monetary'].quantile([.25, .75]); iqr = q3 - q1
n_extreme = int((rfm['Monetary'] > q3 + 3 * iqr).sum())
print(f'Customers with Monetary above Q3 + 3*IQR (kept, log-transformed): {n_extreme:,}')

corr = rfm[FEATS + EXTRA].corr(method='spearman')
display(corr.round(2))
corr.to_csv(P2 / 'spearman_correlation.csv')
plt.figure(figsize=(7, 5.5)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('Spearman correlation of customer features'); plt.tight_layout(); plt.show()

X_rfm = np.log1p(rfm[FEATS].values)
scaler = StandardScaler().fit(X_rfm)
Xs = scaler.transform(X_rfm)
X_ext = np.log1p(rfm[FEATS + EXTRA].values)
scaler_ext = StandardScaler().fit(X_ext)
Xs_ext = scaler_ext.transform(X_ext)

## 3. k-Means clustering: experiments
Grid: number of clusters `k`, initialization (`k-means++` vs `random`), and feature set (`RFM` vs `RFM+` with average order value, tenure and unique products). For each run we record inertia, silhouette, Davies-Bouldin, Calinski-Harabasz, smallest cluster share, and **stability** (Adjusted Rand Index between two different random seeds).

**Selection rule (fixed before looking at results):** among `RFM` runs with `k >= 3` (k = 2 is rarely actionable for marketing), smallest cluster >= 3% of customers and stability ARI >= 0.6, choose the highest silhouette. If none pass, relax the stability condition.

In [ ]:
n = len(Xs)
sil_sample = None if n <= 10000 else 10000

def run_kmeans(Xmat, k, init):
    km = KMeans(n_clusters=k, init=init, n_init=10, random_state=RANDOM_STATE).fit(Xmat)
    km2 = KMeans(n_clusters=k, init=init, n_init=10, random_state=7).fit(Xmat)
    lab = km.labels_
    return km, {
        'inertia': km.inertia_,
        'silhouette': silhouette_score(Xmat, lab, sample_size=sil_sample, random_state=RANDOM_STATE),
        'davies_bouldin': davies_bouldin_score(Xmat, lab),
        'calinski_harabasz': calinski_harabasz_score(Xmat, lab),
        'min_cluster_share': np.bincount(lab).min() / len(lab),
        'stability_ARI': adjusted_rand_score(lab, km2.labels_),
    }

rows = []
for fs, Xmat, ks, inits in [('RFM', Xs, range(2, 11), ['k-means++', 'random']),
                            ('RFM+', Xs_ext, range(3, 9), ['k-means++'])]:
    for k in ks:
        for init in inits:
            _, m = run_kmeans(Xmat, k, init)
            rows.append({'feature_set': fs, 'k': k, 'init': init, **m})
            log_run(f'KM-{fs}-k{k}-{init}', 'clustering', 'KMeans',
                    {'feature_set': fs, 'k': k, 'init': init, 'n_init': 10, 'seed': RANDOM_STATE},
                    'all customers (unsupervised)', **m)
km_df = pd.DataFrame(rows)
display(km_df.round(3))
km_df.to_csv(P2 / 'kmeans_grid.csv', index=False)

cand = km_df[(km_df.feature_set == 'RFM') & (km_df.k >= 3) & (km_df.min_cluster_share >= 0.03) & (km_df.stability_ARI >= 0.6)]
if cand.empty:
    cand = km_df[(km_df.feature_set == 'RFM') & (km_df.k >= 3) & (km_df.min_cluster_share >= 0.03)]
if cand.empty:
    cand = km_df[(km_df.feature_set == 'RFM') & (km_df.k >= 3)]
best = cand.sort_values('silhouette', ascending=False).iloc[0]
BEST_K, BEST_INIT = int(best['k']), best['init']
print(f'Selected: k={BEST_K}, init={BEST_INIT}, silhouette={best.silhouette:.3f}, stability ARI={best.stability_ARI:.3f}')

## 4. Final segmentation and profiling

In [ ]:
final_km = KMeans(n_clusters=BEST_K, init=BEST_INIT, n_init=10, random_state=RANDOM_STATE).fit(Xs)
rfm['Cluster'] = final_km.labels_
overall = rfm[FEATS].median()

prof = rfm.groupby('Cluster').agg(Customers=('CustomerID', 'size'),
                                  Recency_median=('Recency', 'median'),
                                  Frequency_median=('Frequency', 'median'),
                                  Monetary_median=('Monetary', 'median'),
                                  Monetary_total=('Monetary', 'sum'))
prof['Customer_share'] = prof.Customers / prof.Customers.sum()
prof['Revenue_share'] = prof.Monetary_total / prof.Monetary_total.sum()

def name_segment(r):
    recent = r.Recency_median <= overall['Recency']
    freq = r.Frequency_median > overall['Frequency']
    high = r.Monetary_median > overall['Monetary']
    if recent and freq and high: return 'Loyal High-Value'
    if recent and freq: return 'Active Regulars'
    if recent and high: return 'Recent Big Spenders'
    if recent: return 'Recent Low-Frequency'
    if high: return 'At-Risk High-Value'
    return 'Lapsed Low-Value'

names = {c: name_segment(r) for c, r in prof.iterrows()}
from collections import defaultdict
groups = defaultdict(list)
for c, nm in names.items(): groups[nm].append(c)
for nm, cs in groups.items():              # make names unique; order by median spend (highest first)
    if len(cs) > 1:
        ordered = sorted(cs, key=lambda c: -prof.loc[c, 'Monetary_median'])
        for tier, c in enumerate(ordered, 1):
            if nm == 'Loyal High-Value' and tier > 1:
                names[c] = 'Active Regulars' + ('' if tier == 2 else f' (Tier {tier - 1})')   # top spender keeps the 'Loyal' label
            elif nm == 'Loyal High-Value':
                names[c] = nm
            else:
                names[c] = f'{nm} (Tier {tier})'
rfm['Segment'] = rfm['Cluster'].map(names)
prof.insert(0, 'Segment', prof.index.map(names))
display(prof.round(3))
rfm.to_csv(P2 / 'customer_segments.csv', index=False)
prof.to_csv(P2 / 'segment_profile.csv')
print('Segment names are rule-based labels from medians vs overall medians; they are interpretations, not ground truth.')

## 5. Association rules (market-basket analysis, mlxtend)
Baskets = invoices. To keep computation tractable the analysis uses the **150 most frequently purchased products** and invoices with at least 2 of them. We test several minimum-support values and log each run.

**Selection rule:** use the lowest support that keeps frequent itemsets <= 5,000 and still yields at least one rule with lift >= 1.2 and confidence >= 0.3.

In [ ]:
top_codes = tx.groupby('StockCode')['InvoiceNo'].nunique().nlargest(150).index
sub = tx[tx['StockCode'].isin(top_codes)].drop_duplicates(['InvoiceNo', 'StockCode'])
basket = sub.assign(v=True).pivot(index='InvoiceNo', columns='StockCode', values='v').fillna(False).astype(bool)
basket = basket[basket.sum(axis=1) >= 2]
print(f'Baskets used: {len(basket):,} | products: {basket.shape[1]}')
desc = (tx[tx['StockCode'].isin(top_codes)].dropna(subset=['Description'])
        .groupby('StockCode')['Description'].agg(lambda s: s.mode().iat[0]).to_dict())

def make_rules(fi, nb, **kw):
    if fi.empty: return pd.DataFrame()
    try:    return association_rules(fi, num_itemsets=nb, **kw)
    except TypeError: return association_rules(fi, **kw)

results, chosen = {}, None
for sup in [0.03, 0.02, 0.015, 0.01]:
    fi = apriori(basket, min_support=sup, use_colnames=True, max_len=3)
    rl = make_rules(fi, len(basket), metric='lift', min_threshold=1.0)
    good = rl[(rl.lift >= 1.2) & (rl.confidence >= 0.3)] if len(rl) else rl
    results[sup] = (fi, rl, good)
    log_run(f'AR-sup{sup}', 'association_rules', 'Apriori + rules',
            {'min_support': sup, 'max_len': 3, 'metric': 'lift', 'min_threshold': 1.0},
            f'{len(basket)} baskets', itemsets=len(fi), rules=len(rl), rules_lift12_conf30=len(good),
            max_lift=float(rl.lift.max()) if len(rl) else np.nan)
    print(f'support={sup}: itemsets={len(fi):,}, rules={len(rl):,}, strong rules={len(good):,}')
    if len(fi) <= 5000 and len(good) >= 1:
        chosen = sup
AR_SUPPORT = chosen if chosen is not None else 0.01
fi, rl, good = results[AR_SUPPORT]
rules = good.copy() if len(good) else rl.copy()
if len(rules):
    fmt = lambda s: ' + '.join(desc.get(i, i) for i in sorted(s))
    rules['antecedents_txt'] = rules['antecedents'].apply(fmt)
    rules['consequents_txt'] = rules['consequents'].apply(fmt)
    rules = rules.sort_values('lift', ascending=False)
    out = rules[['antecedents_txt', 'consequents_txt', 'support', 'confidence', 'lift']]
    out.to_csv(P2 / 'association_rules.csv', index=False)
    display(out.head(15).round(3))
else:
    pd.DataFrame(columns=['antecedents_txt', 'consequents_txt', 'support', 'confidence', 'lift']).to_csv(P2 / 'association_rules.csv', index=False)
    print('No rules met the thresholds on this data. Report this honestly and consider a larger product set.')
print('Chosen min_support:', AR_SUPPORT)

### Rule network (networkx)
Directed graph: edge = rule `antecedent -> consequent` (single-item rules only for readability). Degree centrality shows which products are most connected.

In [ ]:
G = nx.DiGraph()
single = rules[(rules['antecedents'].apply(len) == 1) & (rules['consequents'].apply(len) == 1)] if len(rules) else rules
for _, r in single.head(60).iterrows():
    G.add_edge(r['antecedents_txt'], r['consequents_txt'], lift=float(r['lift']))
if G.number_of_nodes():
    cent = pd.Series(nx.degree_centrality(G)).sort_values(ascending=False)
    display(cent.head(10).rename('degree_centrality').to_frame().round(3))
    cent.rename('degree_centrality').to_csv(P2 / 'network_centrality.csv')
    plt.figure(figsize=(9, 6)); nx.draw(G, pos=nx.spring_layout(G, seed=RANDOM_STATE), with_labels=False, node_size=60, arrowsize=8); plt.title('Association-rule network (preview)'); plt.show()
else:
    print('Graph is empty (no single-item rules).')
log_run('NET-1', 'network', 'networkx DiGraph', {'source': 'top single-item rules', 'max_rules': 60}, 'rules',
        nodes=G.number_of_nodes(), edges=G.number_of_edges())

## 6. Predictive model: will a customer buy again in the last 90 days?
To avoid **data leakage**, features are built only from purchases **before** a cutoff date (90 days before the reference date). The target is whether the customer purchases at least once **after** the cutoff. Data are split into train (60%), validation (20%) and test (20%), stratified by the target. Models are tuned on validation; the test set is used **once**, for the selected model.

In [ ]:
HORIZON = 90
cutoff = reference_date - pd.Timedelta(days=HORIZON)
hist = tx[tx['InvoiceDate'] < cutoff]; fut = tx[tx['InvoiceDate'] >= cutoff]
PRED_FEATS = ['Recency', 'Frequency', 'Monetary', 'AverageOrderValue', 'CustomerTenure', 'UniqueProducts']
g = hist.groupby('CustomerID').agg(First=('InvoiceDate', 'min'), Last=('InvoiceDate', 'max'),
                                   Frequency=('InvoiceNo', 'nunique'), Monetary=('TotalSales', 'sum'),
                                   UniqueProducts=('StockCode', 'nunique'))
g['Recency'] = (cutoff - g['Last'].dt.normalize()).dt.days
g['AverageOrderValue'] = g['Monetary'] / g['Frequency']
g['CustomerTenure'] = (cutoff - g['First'].dt.normalize()).dt.days
data = g[PRED_FEATS].copy()
data['y'] = data.index.isin(set(fut['CustomerID'])).astype(int)
data = data[(data[PRED_FEATS] >= 0).all(axis=1)]
print(f'Cutoff: {cutoff.date()} | customers with history: {len(data):,} | repeat-buyer rate: {data.y.mean():.1%}')

Xa, ya = data[PRED_FEATS], data['y']
X_tmp, X_te, y_tmp, y_te = train_test_split(Xa, ya, test_size=0.20, stratify=ya, random_state=RANDOM_STATE)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RANDOM_STATE)
print('Train/validation/test sizes:', len(X_tr), len(X_va), len(X_te))

def pipe(clf):
    return Pipeline([('log', FunctionTransformer(np.log1p, validate=False)), ('scale', StandardScaler()), ('clf', clf)])

def metrics(model, X, y):
    prob = model.predict_proba(X)[:, 1]; pred = (prob >= 0.5).astype(int)
    return {'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
            'recall': recall_score(y, pred, zero_division=0), 'f1': f1_score(y, pred, zero_division=0),
            'roc_auc': roc_auc_score(y, prob)}

candidates = [('Baseline (most frequent)', 'DummyClassifier', {'strategy': 'prior'}, pipe(DummyClassifier(strategy='prior')))]
for C in [0.01, 0.1, 1, 10]:
    candidates.append((f'LogReg C={C}', 'LogisticRegression', {'C': C, 'max_iter': 1000},
                       pipe(LogisticRegression(C=C, max_iter=1000, random_state=RANDOM_STATE))))
for d in [3, 5, 8]:
    for leaf in [20, 50]:
        candidates.append((f'Tree depth={d} leaf={leaf}', 'DecisionTree', {'max_depth': d, 'min_samples_leaf': leaf},
                           pipe(DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, random_state=RANDOM_STATE))))
for hl in [(16,), (32, 16)]:
    for a in [1e-4, 1e-2]:
        candidates.append((f'MLP hidden={hl} alpha={a}', 'MLPClassifier', {'hidden': hl, 'alpha': a, 'max_iter': 500},
                           pipe(MLPClassifier(hidden_layer_sizes=hl, alpha=a, max_iter=500, early_stopping=True, random_state=RANDOM_STATE))))

fitted, val_rows = {}, []
for i, (label, model_name, params, mdl) in enumerate(candidates, 1):
    mdl.fit(X_tr, y_tr)
    fitted[label] = mdl
    m = metrics(mdl, X_va, y_va)
    val_rows.append({'candidate': label, 'family': model_name, **m})
    log_run(f'PRED-{i:02d}', 'classification', model_name, params, 'train->validation', **m)
val_df = pd.DataFrame(val_rows).sort_values('roc_auc', ascending=False)
display(val_df.round(3))
val_df.to_csv(P2 / 'predictive_validation_results.csv', index=False)

In [ ]:
best_label = val_df[val_df.family != 'DummyClassifier'].iloc[0]['candidate']
best_model = fitted[best_label]
base_label = 'Baseline (most frequent)'
test_m = metrics(best_model, X_te, y_te); base_m = metrics(fitted[base_label], X_te, y_te)
log_run('PRED-TEST-best', 'classification', val_df.set_index('candidate').loc[best_label, 'family'],
        {'candidate': best_label}, 'test (final, used once)', **test_m)
log_run('PRED-TEST-baseline', 'classification', 'DummyClassifier', {'strategy': 'prior'}, 'test (final, used once)', **base_m)
print('Selected on validation ROC-AUC:', best_label)
display(pd.DataFrame({'selected': test_m, 'baseline': base_m}).T.round(3))

prob_te = best_model.predict_proba(X_te)[:, 1]
pd.DataFrame({'y_true': y_te.values, 'y_prob': prob_te, 'y_pred': (prob_te >= .5).astype(int)}).to_csv(P2 / 'test_predictions.csv', index=False)
print('Confusion matrix [[TN, FP],[FN, TP]]:'); print(confusion_matrix(y_te, (prob_te >= .5).astype(int)))

pi = permutation_importance(best_model, X_te, y_te, scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE)
imp = pd.DataFrame({'feature': PRED_FEATS, 'importance_mean': pi.importances_mean, 'importance_std': pi.importances_std}) \
        .sort_values('importance_mean', ascending=False)
display(imp.round(4)); imp.to_csv(P2 / 'model_importance.csv', index=False)

## 7. Save the experiment log, results summary and model artifacts
`Model_Experiment_Log.xlsx` is the required **Model Experiment Log**; `model_artifacts.pkl` holds the fitted scaler, k-Means model, segment names and the selected predictive pipeline. Reload it with the same scikit-learn version it was saved with.

In [ ]:
log_df = pd.DataFrame(LOG)
log_df.insert(0, 'timestamp', dt.datetime.now().strftime('%Y-%m-%d %H:%M'))
log_df.to_csv(P2 / 'experiment_log.csv', index=False)
with pd.ExcelWriter(P2 / 'Model_Experiment_Log.xlsx') as xw:
    log_df.to_excel(xw, sheet_name='All experiments', index=False)
    for t, d in log_df.groupby('task'):
        d.dropna(axis=1, how='all').to_excel(xw, sheet_name=t[:30], index=False)
print(f'{len(log_df)} experiments logged.')

summary = {
    'n_customers': int(len(rfm)), 'n_lines': int(len(tx)), 'reference_date': str(reference_date.date()),
    'kmeans': {'k': BEST_K, 'init': BEST_INIT, 'silhouette': float(best.silhouette),
               'davies_bouldin': float(best.davies_bouldin), 'calinski_harabasz': float(best.calinski_harabasz),
               'stability_ARI': float(best.stability_ARI), 'min_cluster_share': float(best.min_cluster_share)},
    'association': {'baskets': int(len(basket)), 'min_support': AR_SUPPORT, 'rules_saved': int(len(rules))},
    'predictive': {'cutoff': str(cutoff.date()), 'horizon_days': HORIZON, 'n_customers': int(len(data)),
                   'positive_rate': float(data.y.mean()), 'selected_model': best_label,
                   'test': {k: float(v) for k, v in test_m.items()}, 'baseline_test': {k: float(v) for k, v in base_m.items()},
                   'split_sizes': [int(len(X_tr)), int(len(X_va)), int(len(X_te))]},
    'n_experiments': int(len(log_df)), 'sklearn_version': sklearn.__version__,
}
json.dump(summary, open(P2 / 'results_summary.json', 'w'), indent=2)

artifacts = {'created': dt.datetime.now().isoformat(timespec='seconds'), 'sklearn_version': sklearn.__version__,
             'random_state': RANDOM_STATE, 'rfm_features': FEATS, 'rfm_transform': 'log1p then StandardScaler',
             'scaler': scaler, 'kmeans': final_km, 'segment_names': names,
             'pred_features': PRED_FEATS, 'predictive_model_name': best_label, 'predictive_model': best_model,
             'predictive_cutoff': str(cutoff.date()), 'horizon_days': HORIZON,
             'association_min_support': AR_SUPPORT, 'reference_date': str(reference_date.date())}
with open(P2 / 'model_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

# round-trip check
with open(P2 / 'model_artifacts.pkl', 'rb') as f: chk = pickle.load(f)
assert (chk['kmeans'].predict(chk['scaler'].transform(np.log1p(rfm[FEATS].values))) == rfm['Cluster'].values).all()
print('Artifact reload check passed.')

arc = Path('harborcart_phase2_outputs.zip')
with zipfile.ZipFile(arc, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(P2.iterdir()):
        if p.is_file(): z.write(p, arcname=p.name)
print('Saved', arc, '| files:', ', '.join(p.name for p in sorted(P2.iterdir())))
try:
    from google.colab import files; files.download(str(arc))
except ImportError:
    pass